# FLUKE Coreference Resolution with OpenAI o3 Reasoning Model

This notebook evaluates coreference resolution robustness using OpenAI's o3-2025-04-16 reasoning model with FLUKE linguistic modifications.

In [1]:
from datasets import load_dataset
import dspy
import openai
import os
import re
import pandas as pd
import json
import random
from dotenv import load_dotenv
import glob
import time
from tqdm import tqdm

/Users/hungthinh/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

## Model Configuration

In [3]:
# Available o3 and o1 reasoning models
REASONING_MODELS = {
    'o3-2025-04-16': 'openai/o3-2025-04-16',
    'o1-preview': 'openai/o1-preview',
    'o1-mini': 'openai/o1-mini', 
    'o1': 'openai/o1',
}

# Model selection with different reasoning strategies
REASONING_CONFIGS = {
    'standard': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'standard',
        'description': 'Standard reasoning approach with o3'
    },
    'detailed': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'detailed',
        'description': 'Detailed step-by-step reasoning with o3'
    },
    'efficient': {
        'model': 'o1-mini',
        'instruction_style': 'concise',
        'description': 'Efficient reasoning with o1-mini'
    }
}

# Select configuration
CONFIG_NAME = 'standard'  # Change to 'detailed' or 'efficient'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]
INSTRUCTION_STYLE = config['instruction_style']

print(f"Using configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Instruction style: {INSTRUCTION_STYLE}")
print(f"Description: {config['description']}")

Using configuration: standard
Model: o3-2025-04-16 (openai/o3-2025-04-16)
Instruction style: standard
Description: Standard reasoning approach with o3


In [5]:
# Configure DSPy
lm = dspy.LM(MODEL_ID, temperature=1, max_tokens=5000)
dspy.configure(lm=lm)

## Load Coreference Data

In [9]:
# Load coreference dataset
coref_data = pd.read_json('../../../data/train_dev_test_data/coref/test.json')
coref_samples = coref_data.to_dict('records')[:50]  # Limit for testing

print(f"Loaded {len(coref_samples)} coreference samples")
print(f"Sample: {coref_samples[0]}")

Loaded 50 coreference samples
Sample: {'label': 0, 'candidates': ['Sabina', 'Maria'], 'pronoun': 'she', 'text': 'Sabina is trying to look for Maria, but to no avail, as she worries for her safety.'}


In [ ]:
def reformat_sample(sample):
    """Reformat sample to extract object and span."""
    text = sample['text']
    print(sample)
    sample['text'] = text
    sample['object'] = sample['label']
    sample['span'] = sample['label'][0][1]
    return sample

# Reformat samples
for i, sample in enumerate(coref_samples):
    coref_samples[i] = reformat_sample(sample)

{'label': 0, 'candidates': ['Sabina', 'Maria'], 'pronoun': 'she', 'text': 'Sabina is trying to look for Maria, but to no avail, as she worries for her safety.'}


TypeError: 'int' object is not subscriptable

## Coreference Resolution with o3 Model

In [12]:
class O3Coreference(dspy.Signature):
    """Determine coreference relationships by analyzing pronouns and their referents. Think carefully about the context, grammatical clues, and semantic relationships to identify what the given object refers to. Provide the exact text span that the object refers to."""
    text = dspy.InputField()
    object = dspy.InputField()
    referent = dspy.OutputField(prefix='The object refers to:')

class O3CoreferenceModule(dspy.Module):
    def __init__(self, instruction_style='standard'):
        super().__init__()
        self.instruction_style = instruction_style
        self.prog = dspy.Predict(O3Coreference)

    def forward(self, text, object):
        return self.prog(text=text, object=object)

In [13]:
o3_coref = O3CoreferenceModule(INSTRUCTION_STYLE)

In [ ]:
# Test with a single example
example = coref_samples[0]
print(f"Text: {example['text']}")
print(f"Object: {example['object']}")
print(f"True Referent: {example['span']}")

pred = o3_coref(text=example['text'], object=example['object'])
print(f"\nPredicted Referent: {pred.referent}")

## Evaluation Functions

In [ ]:
def extract_referent(text, true_referent):
    """Extract referent from o3 model output."""
    # Clean the text
    text = text.strip()
    
    # Look for quoted text first
    quoted_matches = re.findall(r'["\']([^"\']*)["\'']', text)
    if quoted_matches:
        return quoted_matches[-1]
    
    # Look for "refers to:" patterns
    patterns = [
        r'refers to:?\s*(.+?)(?:\.|$)',
        r'referent:?\s*(.+?)(?:\.|$)',
        r'span:?\s*(.+?)(?:\.|$)',
        r'answer:?\s*(.+?)(?:\.|$)'
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()
    
    # Return the whole text if no pattern matches
    return text

def coref_eval_metric(true_referent, predicted_referent):
    """Evaluate coreference prediction."""
    if not predicted_referent:
        return False
        
    # Clean and normalize
    true_clean = true_referent.lower().strip()
    pred_clean = predicted_referent.lower().strip()
    
    # Exact match
    if true_clean == pred_clean:
        return True
    
    # Partial match (predicted contains true or vice versa)
    if true_clean in pred_clean or pred_clean in true_clean:
        return True
    
    return False

## Evaluate Original Dataset

In [ ]:
# Evaluate original samples
original_results = []

print(f"Evaluating {len(coref_samples)} original samples...")

for i, sample in enumerate(tqdm(coref_samples)):
    try:
        pred = o3_coref(text=sample['text'], object=sample['object'])
        predicted_referent = extract_referent(pred.referent, sample['span'])
        correct = coref_eval_metric(sample['span'], predicted_referent)
        
        result = {
            'text': sample['text'],
            'object': sample['object'],
            'true_referent': sample['span'],
            'predicted_referent': predicted_referent,
            'correct': correct,
            'raw_output': pred.referent
        }
        original_results.append(result)
        
        # Add delay for rate limiting
        time.sleep(2)
        
    except Exception as e:
        print(f"Error processing sample {i}: {e}")
        continue

In [ ]:
# Calculate accuracy
correct_count = sum(1 for r in original_results if r['correct'])
accuracy = correct_count / len(original_results) if original_results else 0

print(f"Original Dataset Results:")
print(f"Samples: {len(original_results)}")
print(f"Correct: {correct_count}")
print(f"Accuracy: {accuracy:.3f}")

# Save results
df_original = pd.DataFrame(original_results)
output_file = f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv'
df_original.to_csv(output_file, index=False)
print(f"Results saved to: {output_file}")

## Evaluate Modified Datasets

In [ ]:
# Test with a subset of modifications
test_modifications = [
    'negation_100.json',
    'capitalization_100.json', 
    'punctuation_100.json',
    'active_to_passive_100.json'
]

json_files = glob.glob('../data/modified_data/coref/*_100.json')
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"Testing modifications: {[f.split('/')[-1] for f in json_files]}")

In [ ]:
for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    # Load modification data
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Limit samples for testing
    data = data[:15]
    
    modification_results = []
    
    for i, sample in enumerate(tqdm(data)):
        try:
            # Get the modification details
            modified_text = sample['modified_text']
            original_text = sample['original_text']
            object_ref = sample['object']
            true_referent = sample['refer_to']
            
            # Predict on modified text
            pred = o3_coref(text=modified_text, object=object_ref)
            predicted_referent = extract_referent(pred.referent, true_referent)
            correct = coref_eval_metric(true_referent, predicted_referent)
            
            result = {
                'original_text': original_text,
                'modified_text': modified_text,
                'object': object_ref,
                'true_referent': true_referent,
                'predicted_referent': predicted_referent,
                'correct': correct,
                'modification_type': sample.get('type', 'unknown'),
                'raw_output': pred.referent
            }
            modification_results.append(result)
            
            time.sleep(2)  # Rate limiting
            
        except Exception as e:
            print(f"Error processing sample {i}: {e}")
            continue
    
    # Calculate accuracy for this modification
    if modification_results:
        mod_correct = sum(1 for r in modification_results if r['correct'])
        mod_accuracy = mod_correct / len(modification_results)
        print(f"Accuracy: {mod_accuracy:.3f} ({mod_correct}/{len(modification_results)})")
        
        # Save results
        df_mod = pd.DataFrame(modification_results)
        mod_name = json_file.split('/')[-1].replace('.json', '')
        output_file = f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
        df_mod.to_csv(output_file, index=False)
        print(f"Saved: {output_file}")
    else:
        print("No results to save")

## Analysis and Comparison

In [ ]:
# Aggregate results across modifications
result_files = glob.glob(f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')
aggregated_results = []

for file in result_files:
    mod_type = file.split('-')[-1].replace('.csv', '')
    df = pd.read_csv(file)
    
    accuracy = df['correct'].mean() if 'correct' in df.columns else 0
    total_samples = len(df)
    
    aggregated_results.append({
        'modification': mod_type,
        'accuracy': round(accuracy, 3),
        'samples': total_samples,
        'model_config': f'{MODEL_NAME}-{CONFIG_NAME}'
    })

if aggregated_results:
    agg_df = pd.DataFrame(aggregated_results)
    print("\nAggregated Results:")
    print(agg_df)
    
    # Save aggregated results
    agg_df.to_csv(f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-aggregated.csv', index=False)

In [ ]:
# Show some example reasoning outputs
if original_results:
    print("\nSample o3 Reasoning Outputs:")
    for i, result in enumerate(original_results[:3]):
        print(f"\nExample {i+1}:")
        print(f"Text: {result['text'][:100]}...")
        print(f"Object: {result['object']}")
        print(f"True: {result['true_referent']}")
        print(f"Predicted: {result['predicted_referent']}")
        print(f"Correct: {result['correct']}")
        print(f"Raw reasoning: {result['raw_output'][:400]}...")

In [ ]:
print(f"\nEvaluation complete for {MODEL_NAME} with {CONFIG_NAME} configuration!")
print(f"Configuration used: {config['description']}")
print(f"Files saved in results/coref/ with prefix '{MODEL_NAME}-{CONFIG_NAME}-'")

if original_results:
    base_acc = sum(1 for r in original_results if r['correct']) / len(original_results)
    print(f"Base accuracy: {base_acc:.3f}")

print(f"\nKey insights with o3 model:")
print(f"- o3 provides enhanced reasoning for coreference resolution")
print(f"- Advanced reasoning capabilities may improve robustness")
print(f"- Detailed reasoning traces help understand model decisions")
print(f"- Results show o3's performance on various linguistic modifications")